# Dataset exploration

In [1]:
import pandas as pd

X = pd.read_csv("../data/X_train_6ZIKlTY.csv")
Y = pd.read_csv("../data/y_train_lXj6X5y.csv")



In [2]:
eda = pd.concat([X, Y], axis=1)
print(eda.info())

<class 'pandas.DataFrame'>
RangeIndex: 55603 entries, 0 to 55602
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Index                  55603 non-null  int64  
 1   patient_id             55603 non-null  str    
 2   cohort                 55603 non-null  str    
 3   sexM                   55603 non-null  int64  
 4   gene                   37645 non-null  str    
 5   age_at_diagnosis       52773 non-null  float64
 6   age                    55603 non-null  float64
 7   ledd                   35010 non-null  float64
 8   time_since_intake_on   29663 non-null  float64
 9   time_since_intake_off  11775 non-null  float64
 10  on                     38944 non-null  float64
 11  off                    32196 non-null  float64
 12  Index                  55603 non-null  int64  
 13  target                 55603 non-null  float64
dtypes: float64(8), int64(3), str(3)
memory usage: 5.9 MB
None


# Dataset minimum processing

In [3]:
# ====== DATASET SPLIT ======
from sklearn.model_selection import GroupShuffleSplit

y = Y["target"]
groups = X["patient_id"]
features = X.copy()

split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(split.split(features, y, groups=groups))

X_train = features.iloc[train_idx].copy()
X_val = features.iloc[val_idx].copy()
y_train = y.iloc[train_idx]
y_val = y.iloc[val_idx]


# ====== ENCODING CATEGORICAL VARIABLES ======
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

exclude = ["patient_id"]
categorical = ["cohort", "gene"]

encoder = ColumnTransformer(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical)],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X_train_encoded = encoder.fit_transform(X_train)
X_val_encoded = encoder.transform(X_val)


In [4]:
# ===== IMPUTATION ======
# We will impute missing values with the median of the training set and create a flag for each feature indicating whether it was missing or not.
cols = X_train_encoded.columns[X_train_encoded.isna().any()]
medians = X_train_encoded[cols].median()

def impute(df):
    flags = df[cols].isna().astype(int).add_suffix("_missing")
    return pd.concat([df.fillna(medians), flags], axis=1)

X_train_imputed = impute(X_train_encoded)
X_val_imputed = impute(X_val_encoded)

In [5]:
print(X_train_imputed.columns)
print(X_train_imputed.isna().sum())


Index(['cohort_A', 'cohort_B', 'gene_GBA+', 'gene_LRRK2+', 'gene_No Mutation',
       'gene_OTHER+', 'gene_nan', 'Index', 'patient_id', 'sexM',
       'age_at_diagnosis', 'age', 'ledd', 'time_since_intake_on',
       'time_since_intake_off', 'on', 'off', 'age_at_diagnosis_missing',
       'ledd_missing', 'time_since_intake_on_missing',
       'time_since_intake_off_missing', 'on_missing', 'off_missing'],
      dtype='str')
cohort_A                         0
cohort_B                         0
gene_GBA+                        0
gene_LRRK2+                      0
gene_No Mutation                 0
gene_OTHER+                      0
gene_nan                         0
Index                            0
patient_id                       0
sexM                             0
age_at_diagnosis                 0
age                              0
ledd                             0
time_since_intake_on             0
time_since_intake_off            0
on                               0
off          

In [6]:
# ====== FINAL DATASET FOR MODELING ======
X_train_model = X_train_imputed.drop(columns=["Index", "patient_id"])
X_val_model = X_val_imputed.drop(columns=["Index", "patient_id"])

In [15]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor


models = {
    "LR": make_pipeline(StandardScaler(), LinearRegression()),
    "XGBoost": XGBRegressor(random_state=42),
    #"SVM": make_pipeline(StandardScaler(), SVR()),
    "HistGB": HistGradientBoostingRegressor(random_state=42),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(X_train_model, y_train)
    print(name, root_mean_squared_error(y_val, model.predict(X_val_model)))

LR 8.526105809739494
XGBoost 7.498791526723528
HistGB 7.401357196655132
RandomForest 7.663463614728675
